# 🚀 Huấn Luyện Qwen 2.5 7B NLU v2 Với Unsloth & Google Drive
### Dự án: Travel AI Planner Việt Nam
- **Mục tiêu**: Huấn luyện mô hình NLU `v2` trên bộ dữ liệu thuần đối kháng (`train_nlu_v2_pure.jsonl`) để giải quyết triệt để lỗi đảo ngữ (Ví dụ: *"Tới Quảng Nam ... xuất phát từ Bắc Ninh"*).
- **Công nghệ**: **Unsloth (QLoRA 4-bit)** giúp train nhanh gấp 2 lần, tiết kiệm 70% VRAM trên GPU T4 Colab miễn phí.
- **Bảo toàn dữ liệu**: **Kết nối Google Drive vĩnh viễn** — tự động lưu checkpoint từng bước (`save_steps`) và xuất file `.gguf` trực tiếp vào Drive, không lo mất file khi Colab ngắt kết nối.

## 1. Kết Nối Google Drive & Thiết Lập Thư Mục Lưu Trữ

In [ ]:
# ==============================================================================
# BƯỚC 1: GẮN KẾT GOOGLE DRIVE & KHỞI TẠO ĐƯỜNG DẪN DỰ ÁN
# ==============================================================================
import os
from google.colab import drive

# 1. Gắn kết Drive (Bấm nút cấp quyền trong popup)
drive.mount('/content/drive')

# 2. Khai báo các đường dẫn lưu trữ vĩnh viễn trên Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/vn_travel_qwen_v2"
CHECKPOINTS_DIR = os.path.join(DRIVE_PROJECT_DIR, "checkpoints")
LORA_OUTPUT_DIR = os.path.join(DRIVE_PROJECT_DIR, "lora_weights")
GGUF_OUTPUT_DIR = os.path.join(DRIVE_PROJECT_DIR, "gguf")

os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(LORA_OUTPUT_DIR, exist_ok=True)
os.makedirs(GGUF_OUTPUT_DIR, exist_ok=True)

print("✅ Đã kết nối Google Drive thành công!")
print(f"📁 Thư mục dự án: {DRIVE_PROJECT_DIR}")
print(f"💾 Thư mục lưu checkpoint: {CHECKPOINTS_DIR}")
print(f"📦 Thư mục lưu file GGUF xuất xưởng: {GGUF_OUTPUT_DIR}")

## 2. Cài Đặt Unsloth & Các Thư Viện Tăng Tốc

In [ ]:
# ==============================================================================
# BƯỚC 2: CÀI ĐẶT UNSLOTH CHO GOOGLE COLAB (GPU T4)
# ==============================================================================
%%capture
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 3. Khởi Tạo Model Qwen 2.5 7B & Gắn LoRA Adapter

In [ ]:
# ==============================================================================
# BƯỚC 3: LOAD MODEL QWEN 2.5 7B INSTRUCT 4-BIT & THIẾT LẬP LORA
# ==============================================================================
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Đảm bảo chứa trọn vẹn System prompt, User prompt và JSON output
dtype = None         # Tự động chọn Float16 (cho T4) hoặc Bfloat16 (cho Ampere/A100)
load_in_4bit = True  # Giảm 4-bit giúp chỉ tốn ~6GB VRAM, chạy cực mượt trên GPU T4 miễn phí

# Tải pre-quantized 4-bit model chính chủ Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Gắn LoRA Adapters trên toàn bộ các Linear Modules quan trọng
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("✅ Khởi tạo Qwen 2.5 7B + LoRA thành công!")

## 4. Nạp Dữ Liệu Huấn Luyện v2 Thuần Mới (`train_nlu_v2_pure.jsonl`)

In [ ]:
# ==============================================================================
# BƯỚC 4: NẠP DỮ LIỆU & ÁP DỤNG QWEN CHAT TEMPLATE
# ==============================================================================
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Áp dụng Chat Template chuẩn của Qwen 2.5
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

DATASET_DRIVE_PATH = os.path.join(DRIVE_PROJECT_DIR, "train_nlu_master_v3.jsonl")
LOCAL_DATASET_PATH = "/content/train_nlu_master_v3.jsonl"

dataset_path = None
if os.path.exists(DATASET_DRIVE_PATH):
    dataset_path = DATASET_DRIVE_PATH
    print(f"📖 Đã tìm thấy dataset trên Drive: {dataset_path}")
elif os.path.exists(LOCAL_DATASET_PATH):
    dataset_path = LOCAL_DATASET_PATH
    print(f"📖 Đã tìm thấy dataset tải lên Colab: {dataset_path}")
else:
    print("⚠️ Chưa thấy dataset trên Drive! Vui lòng tải file 'train_nlu_master_v3.jsonl' lên:")
    from google.colab import files
    uploaded = files.upload()
    uploaded_file = list(uploaded.keys())[0]
    # Tự động sao chép sang Drive để các lần chạy sau không phải upload lại
    !cp "{uploaded_file}" "{DATASET_DRIVE_PATH}"
    dataset_path = DATASET_DRIVE_PATH
    print(f"✅ Đã tự động lưu dataset vào Drive: {DATASET_DRIVE_PATH}")

# Đọc các bản ghi jsonl
raw_records = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))

print(f"📊 Số lượng mẫu đối kháng v2 thuần: {len(raw_records)} mẫu")

# Format text theo Qwen Chat Template
def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False) for conv in convs]
    return { "text" : texts }

dataset = Dataset.from_list(raw_records)
dataset = dataset.map(formatting_prompts_func, batched = True)
print("✅ Hoàn tất chuyển đổi dữ liệu sang format huấn luyện!")

## 5. Huấn Luyện QLoRA & Tự Động Lưu Checkpoint Từng Bước Vào Google Drive
- Cứ mỗi `save_steps = 15` bước, hệ thống sẽ tự động lưu 1 checkpoint vào Google Drive.
- Nếu Colab bị ngắt kết nối giữa chừng, khi chạy lại ô này code sẽ **tự động resume** từ checkpoint gần nhất!

In [ ]:
# ==============================================================================
# BƯỚC 5: HUẤN LUYỆN VÀ TỰ ĐỘNG LƯU BƯỚC (SAVE STEPS) VÀO DRIVE
# ==============================================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Quét tìm checkpoint cũ trên Drive (nếu có để tự động resume)
last_checkpoint = None
if os.path.exists(CHECKPOINTS_DIR):
    ckpts = [os.path.join(CHECKPOINTS_DIR, d) for d in os.listdir(CHECKPOINTS_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        ckpts.sort(key=lambda x: int(x.split("-")[-1]))
        last_checkpoint = ckpts[-1]
        print(f"🔄 Phát hiện checkpoint có sẵn trên Drive: {last_checkpoint}")
        print("➡️ Tiến trình sẽ tự động tiếp tục (RESUME) từ checkpoint này!")

# 2. Cấu hình tham số huấn luyện
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4, # Effective batch size = 8
    warmup_steps = 5,
    num_train_epochs = 3,            # Huấn luyện 3 epochs cho 128 mẫu đối kháng
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    # === LƯU BƯỚC VÀO GOOGLE DRIVE ===
    output_dir = CHECKPOINTS_DIR,     # Lưu trực tiếp vào Google Drive!
    save_strategy = "steps",
    save_steps = 15,                  # Cứ mỗi 15 bước lưu 1 checkpoint
    save_total_limit = 2,             # Giữ lại 2 checkpoint mới nhất, tránh tốn dung lượng Drive
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

print("🚀 Bắt đầu quá trình huấn luyện Qwen 2.5 7B...")
if last_checkpoint:
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

print(f"🎉 Huấn luyện hoàn thành! Các checkpoint đã được lưu an toàn tại: {CHECKPOINTS_DIR}")

## 6. Xuất File GGUF (q4_k_m) Cực Chuẩn Vào Google Drive
- **Lưu ý kiến trúc quan trọng**: Quá trình merge 16-bit safetensors ngốn ~15GB tạm thời. Nếu ghi trực tiếp vào Google Drive sẽ bị nghẽn mạng FUSE hoặc hết dung lượng Drive 15GB.
- **Giải pháp chuẩn**: Merge và convert trên đĩa SSD siêu tốc cục bộ của Colab (`/content/local_gguf`), sau đó copy **duy nhất file .gguf (~4.5GB)** sang Google Drive!

In [ ]:
# ==============================================================================
# BƯỚC 6: XUẤT FILE GGUF Q4_K_M VÀ LƯU VÀO GOOGLE DRIVE
# ==============================================================================
import os
import glob
import shutil
from unsloth import FastLanguageModel

# 1. Dọn dẹp các file safetensors tạm trên Drive (nếu có từ lần chạy trước gây đầy dung lượng)
bad_drive_gguf = "/content/drive/MyDrive/vn_travel_qwen_v2/gguf"
if os.path.exists(bad_drive_gguf):
    print("🧹 Đang dọn dẹp thư mục tạm cũ trên Drive để giải phóng dung lượng...")
    !rm -rf "{bad_drive_gguf}"

# 2. Kiểm tra biến model trong bộ nhớ. Nếu kernel vừa restart, tải lại nhanh từ LoRA weights đã lưu
if 'model' not in locals() or model is None:
    print("🔄 Kernel đã restart, đang nạp lại model từ LoRA weights trên Google Drive...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = LORA_OUTPUT_DIR,
        max_seq_length = 2048,
        load_in_4bit = True,
    )

# 3. Lưu trọng số LoRA vào Drive (đã lưu ở bước trước, lưu lại để bảo đảm 100%)
print("💾 1/3: Đảm bảo LoRA weights đã lưu an toàn trên Google Drive...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"✅ LoRA weights an toàn tại: {LORA_OUTPUT_DIR}")

# 4. Thực hiện Merge & Quantize GGUF trên đĩa SSD cục bộ Colab (/content/local_gguf)
# Đĩa local Colab có >100GB trống, tốc độ cực nhanh, không bị lỗi FUSE hay tràn dung lượng Drive!
LOCAL_GGUF_DIR = "/content/local_gguf"
os.makedirs(LOCAL_GGUF_DIR, exist_ok=True)

print("\n📦 2/3: Đang convert và quantize sang GGUF q4_k_m trên SSD cục bộ (mất ~3-5 phút)...\n")
model.save_pretrained_gguf(
    LOCAL_GGUF_DIR,
    tokenizer,
    quantization_method = "q4_k_m"
)

# 5. Sao chép file GGUF hoàn thiện (~4.5GB) sang Google Drive
print("\n🚀 3/3: Đang sao chép file GGUF hoàn thiện sang Google Drive...")
DRIVE_GGUF_FINAL = "/content/drive/MyDrive/vn_travel_qwen_v2/Qwen2.5-7B-Instruct-v2.Q4_K_M.gguf"

local_gguf_files = glob.glob(f"{LOCAL_GGUF_DIR}/*q4_k_m*.gguf") or glob.glob(f"{LOCAL_GGUF_DIR}/*Q4_K_M*.gguf")
if not local_gguf_files:
    local_gguf_files = [f for f in glob.glob(f"{LOCAL_GGUF_DIR}/*.gguf") if "f16" not in f.lower()]

if local_gguf_files:
    target_local_file = local_gguf_files[0]
    print(f"📄 Tìm thấy file GGUF: {target_local_file} ({os.path.getsize(target_local_file) / (1024**3):.2f} GB)")
    shutil.copy2(target_local_file, DRIVE_GGUF_FINAL)
    print(f"🎉 THÀNH CÔNG RỰC RỠ! File đã được lưu vào Drive:")
    print(f"📁 {DRIVE_GGUF_FINAL}")
    !ls -lh "{DRIVE_GGUF_FINAL}"
else:
    print("⚠️ Không tìm thấy file q4_k_m, danh sách file trong local_gguf:")
    !ls -lh "{LOCAL_GGUF_DIR}"


## 7. Thử Nghiệm Trực Tiếp (Inference Test) Ngay Trên Colab
Kiểm tra ngay ca kinh điển: **"Tới Quảng Nam đi bằng máy bay xuất phát từ Bắc Ninh"**

In [ ]:
# ==============================================================================
# BƯỚC 7: TEST INFERENCE NGAY TRÊN COLAB
# ==============================================================================
FastLanguageModel.for_inference(model)

test_prompt = "Tới Quảng Nam đi bằng máy bay xuất phát từ Bắc Ninh 3 ngày"

test_messages = [
    {
        "role": "system",
        "content": "Bạn là NLU Engine chuyên dụng cho hệ thống Travel AI Việt Nam.\nNhiệm vụ: Phân tích yêu cầu của người dùng và trích xuất thành định dạng JSON chuẩn.\nQuy tắc:\n1. Chỉ trả về duy nhất chuỗi JSON hợp lệ bắt đầu bằng '{' và kết thúc bằng '}', không kèm markdown hay lời dẫn.\n2. Xác định đúng intent trong các loại: 'plan_itinerary', 'search_place', 'ask_advisory', 'clarify_needed'.\n3. Nếu người dùng không đề cập thông tin nào, trường tương ứng BẮT BUỘC để null (hoặc [] với mảng). Tuyệt đối không tự suy diễn số liệu."
    },
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True, temperature = 0.0)
response = tokenizer.batch_decode(outputs)
answer = response[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

print("\n================ [KẾT QUẢ PARSE CỦA MODEL V2] ================")
print(answer)
print("===============================================================")